<a href="https://colab.research.google.com/github/yse-eds-cert/code-along-notebooks/blob/main/live-sessions/M4_nyt_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Before you start, check runtime!

Google Colab defaults to a Python runtime. Because this notebook is written in R, you need to switch the runtime once before running any code:

1. In the Colab menu bar, click **Runtime → Change runtime type**.
2. Under **Runtime type**, select **R**.
3. Click **Save**. Colab will reload the notebook with R.

# Recreating a New York Times Graph

In this live session, we'll recreate a chart from the New York Times article *"The Last 8 Years Were the Hottest on Record"*: **"Global average temperature, compared with mid-20th century"** (Graph 6 in the article).

Recreating a published graph is a great way to practice data visualization. You have to find the data, read it in, clean it, and then build up the plot piece by piece until it matches the original.

**Data:** NASA GISS Surface Temperature Analysis (GISTEMP v4), https://data.giss.nasa.gov/gistemp/

GISTEMP reports temperature *anomalies*: how much warmer or cooler each year was than the 1951-1980 average. That 1951-1980 average is the "mid-20th century" baseline the NYT chart uses, so we can plot the anomalies as they are. No re-baselining needed.

## Learning Objectives:
- Read a CSV directly from a URL, including a file with a couple of formatting quirks
- Clean data with `select()`, `rename()`, `drop_na()`, and `mutate()`
- Build a `ggplot` one layer at a time
- Customize colors, labels, and the theme to match a published graphic

Let's get started by loading the tidyverse!

In [ ]:
library(tidyverse)

# 1. Pull the data

We'll use the "Global-mean monthly, seasonal, and annual means" CSV (Land-Ocean Temperature Index) linked from the GISTEMP data page. `read_csv()` can read a file straight from a URL, so there's nothing to download.

This file has two quirks worth pointing out:
- **Line 1 is a title** ("Land-Ocean: Global Means"), not the column header. `skip = 1` tells `read_csv()` to skip that line.
- **Missing values are coded as `***`** (and sometimes `****`), not left blank. `na = c("***", "****")` tells `read_csv()` to treat these as missing. Without it, any column containing `***` would be read in as text (character) instead of numbers!

In [ ]:
url <- "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"

raw <- read_csv(url, skip = 1, na = c("***", "****"))

# If the download fails (firewall, etc.), download the CSV by hand, upload it to
# Colab (folder icon in the left sidebar), and use:
# raw <- read_csv("GLB.Ts+dSST.csv", skip = 1, na = c("***", "****"))

Let's take a look at what we read in. Each row is a year, with a column for each month, a few seasonal averages, and the annual average.

In [ ]:
glimpse(raw)

# 2. Clean the data

We only need two columns: the year and the annual mean. In GISTEMP, the annual (January-December) mean lives in a column called `J-D`. The hyphen makes it a *non-syntactic* name (R would read `J-D` as "J minus D"), so we need to wrap it in backticks: `` `J-D` ``.

`drop_na(anomaly)` drops any year without a full annual average. That's the current year, since it isn't over yet.

Last, we add a `category` column that we'll use to color the bars. We'll match the NYT's colors, then add one more color for the years since the article came out (the NYT chart ends in 2022):
- **cool** (blue): years below the mid-20th century average
- **warm** (light orange): years above the average
- **last8** (dark orange): 2015-2022, the "last 8 years" from the NYT headline
- **last4** (dark red): 2023 on, the years since the NYT article

We do this in three steps inside `mutate()`: first we label every year `"cool"` or `"warm"`, then we overwrite 2015-2022 as `"last8"`, and then we overwrite 2023 on as `"last4"`.

In [ ]:
temps <- raw %>%
  select(Year, `J-D`) %>%                    # keep only the two columns we need
  rename(year = Year, anomaly = `J-D`) %>%   # give them friendlier names
  drop_na(anomaly) %>%                       # drop years without a full annual average
  mutate(
    # Step 1: blue if below average, light orange if above
    category = ifelse(anomaly >= 0, "warm", "cool"),
    # Step 2: overwrite 2015-2022 as dark orange
    category = ifelse(year >= 2015 & year <= 2022, "last8", category),
    # Step 3: overwrite the years since the NYT article (2023 on) as dark red
    category = ifelse(year > 2022 & year <= 2026, "last4", category)
  )

temps

### Sanity check

`slice_max(year)` keeps the row with the largest value of `year`, which is the most recent full year in the data. Saving it as an object means we can use its value later (for example, to label it) instead of hardcoding the number.

In [ ]:
# Pull out the final year's value so we can label it (instead of hardcoding)
last_yr <- temps %>% slice_max(year)

last_yr

The NYT chart ends in 2022 and reports that year as **+0.89°C**. Our data includes more recent years, so to check our numbers against the NYT, let's filter to 2022. (NASA revises GISTEMP periodically, so your value may differ by ~0.01.)

In [ ]:
# Quick sanity check: NYT reports 2022 as +0.89 C
temps %>% filter(year == 2022)

# 3. Plot

Before we build the plot, let's set up a few objects it will use.

**Colors.** This is a *named* vector of hex color codes. The names (`cool`, `warm`, `last8`, `last4`) match the values in our `category` column, which is how `ggplot` will know which color goes with which bars.

In [ ]:
nyt_colors <- c(
  cool   = "#a3d4e4",
  warm   = "#f6c182",
  last8  = "#ee8539",
  last4  = "#d73027"
)

**Reference lines.** In the NYT chart, these are *white* lines drawn on top of the bars, so they're only visible where they cut through a bar.

In [ ]:
# Reference levels
ref_lines <- c(-0.25, 0.25, 0.50, 0.75)

**Reference line labels.** We place these by hand with a small data frame: one row per label, with its x and y position and its text. `hjust` and `vjust` fine-tune the placement:
- `hjust = 1` right-aligns the text, so it *ends* at `my_x`. `hjust = 0` left-aligns it, so it *starts* at `my_x`.
- A negative `vjust` (like `-0.4`) nudges the text up so it sits just above its line. `vjust = 0.5` centers the text on its line.

In [ ]:
# Hand-placed labels for the reference lines (x = where the label ends)
ref_labels <- data.frame(
  my_x     = c(1938, 1976, 1986, 2005),
  my_y     = c(-0.25, 0.25, 0.50, 0.75),    # same as the reference lines
  my_label = c("-0.25°C", "+0.25°C", "+0.50°C", "+0.75°C"),
  my_hjust = c(0, 1, 1, 1),                 # way to adjust horizontal placement
  my_vjust = c(0.5, -0.4, -0.4, -0.4)       # way to adjust vertical placement
)

ref_labels

## Building the plot, one layer at a time

A `ggplot` is built in layers, and each `+` adds one more piece. To see what each piece of code does, we'll save the plot as an object after each step (`p1`, `p2`, ...) and add one layer at a time. At the end, we'll put all the code together in a single cell.

### Step 1: A bar chart

Each bar is one year, and its height is that year's temperature anomaly. We use `geom_col()` because the bar heights are values already in our data (`geom_bar()`, by contrast, *counts* rows and works as a histogram). `fill = category` colors each bar by its category, and `width = 1` makes the bars touch, with no gaps between them.

Don't worry about the colors yet. These are ggplot's defaults, and we'll swap in the NYT colors in Step 6.

In [ ]:
# I want a bar chart, each bar is 1 year, and the fill color of each bar is temps$category
p1 <- ggplot(temps, aes(x = year, y = anomaly)) +
  geom_col(aes(fill = category), width = 1)

p1

### Step 2: Reference lines

`geom_hline()` draws horizontal lines at each value in `ref_lines`. For now, they blend in with ggplot's gray background and white gridlines, but you can see them cutting through the bars. They'll stand out once we switch to a blank theme in Step 8.

In [ ]:
# White reference lines on top of the bars
p2 <- p1 +
  geom_hline(yintercept = ref_lines, color = "white", linewidth = 0.5)

p2

### Step 3: Zero line

A gray line at zero marks the baseline (the 1951-1980 average). Bars above it are years warmer than the mid-20th century, and bars below it are cooler.

In [ ]:
# Gray zero (baseline) line
p3 <- p2 +
  geom_hline(yintercept = 0, color = "grey45", linewidth = 0.5)

p3

### Step 4: Label the reference lines

`geom_text()` adds the labels from `ref_labels`. Two arguments to notice:
- `data = ref_labels` tells this layer to use a *different* data frame than the rest of the plot.
- `inherit.aes = FALSE` tells this layer *not* to use the plot's `aes(x = year, y = anomaly)`, since `ref_labels` doesn't have those columns.

In [ ]:
# Reference line labels
p4 <- p3 +
  geom_text(
    data = ref_labels,
    aes(x = my_x, y = my_y, label = my_label, hjust = my_hjust, vjust = my_vjust),
    color = "grey45", size = 4.2, inherit.aes = FALSE
  )

p4

### Step 5: Title, subtitle, and caption

`labs()` sets the plot's text. Setting `x = NULL` and `y = NULL` removes the axis titles, since the NYT chart doesn't have them.

In [ ]:
# Add title, subtitle, and caption
p5 <- p4 +
  labs(
    title = "Global average temperature",
    subtitle = "Compared with mid-20th century",
    caption = "Source: NASA Goddard Institute for Space Studies",
    x = NULL,
    y = NULL
  )

p5

### Step 6: NYT colors

`scale_fill_manual()` swaps the default fill colors for our `nyt_colors`. `guide = "none"` removes the legend, since the NYT chart doesn't have one.

In [ ]:
# Change fill colors and get rid of the legend
p6 <- p5 +
  scale_fill_manual(values = nyt_colors, guide = "none")

p6

### Step 7: X-axis labels

`scale_x_continuous(breaks = ...)` controls where the x-axis labels go. `seq(1880, 2020, by = 20)` creates the vector `1880, 1900, 1920, ..., 2020`.

In [ ]:
# Label x-axis ticks every 20 years
p7 <- p6 +
  scale_x_continuous(breaks = seq(1880, 2020, by = 20))

p7

### Step 8: Theme

Last, the overall look. `theme_minimal()` gives us a clean starting point (`base_size = 13` sets the default text size), and `theme()` removes or restyles individual pieces:
- `panel.grid = element_blank()` removes all gridlines
- `axis.text.y = element_blank()` removes the y-axis numbers (our reference line labels do that job instead)
- The large negative `vjust` values push the title and subtitle *down* into the empty upper-left area of the plot, like the NYT
- The caption is gray and left-aligned (`hjust = 0`)

In [ ]:
# Start from a blank canvas and add back only what the NYT shows (x-axis year labels)
p8 <- p7 +
  theme_minimal(base_size = 13) +
  theme(
    panel.grid    = element_blank(),     # remove all gridlines
    axis.text.y   = element_blank(),     # remove y-axis numbers
    plot.title    = element_text(face = "bold", vjust = -40),
    plot.subtitle = element_text(vjust = -47),
    plot.caption  = element_text(color = "grey50", hjust = 0)
  )

p8

## All together

Here is the full plot code in one cell. Once you know what you want, this is how you'd normally write it.

Compare it with the original NYT graphic. What matches? What's different?

In [ ]:
ggplot(temps, aes(x = year, y = anomaly)) +

  # I want a bar chart, each bar is 1 year, and the fill color of each bar is temps$category
  geom_col(aes(fill = category), width = 1) +

  # White reference lines on top of the bars
  geom_hline(yintercept = ref_lines, color = "white", linewidth = 0.5) +

  # Gray zero (baseline) line
  geom_hline(yintercept = 0, color = "grey45", linewidth = 0.5) +

  # Reference line labels
  geom_text(
    data = ref_labels,
    aes(x = my_x, y = my_y, label = my_label, hjust = my_hjust, vjust = my_vjust),
    color = "grey45", size = 4.2, inherit.aes = FALSE
  ) +

  # Add title, subtitle, and caption
  labs(
    title = "Global average temperature",
    subtitle = "Compared with mid-20th century",
    caption = "Source: NASA Goddard Institute for Space Studies",
    x = NULL,
    y = NULL
  ) +

  # Change fill colors and get rid of the legend
  scale_fill_manual(values = nyt_colors, guide = "none") +

  # Label x-axis ticks every 20 years
  scale_x_continuous(breaks = seq(1880, 2020, by = 20)) +

  # Start from a blank canvas and add back only what the NYT shows (x-axis year labels)
  theme_minimal(base_size = 13) +
  theme(
    panel.grid    = element_blank(),     # remove all gridlines
    axis.text.y   = element_blank(),     # remove y-axis numbers
    plot.title    = element_text(face = "bold", vjust = -40),
    plot.subtitle = element_text(vjust = -47),
    plot.caption  = element_text(color = "grey50", hjust = 0)
  )